# 🎯 Flight missions

Write and test drone missions here. MAVLink comes from SITL straight to Windows (UDP:14550) — no WSL terminals needed.

**Before you start**: in the `drone_control_ardupilot.ipynb` notebook run "BRIDGE" (`bridge_up`) and "SITL" (`launch_sitl`). Firewall UDP 14550 must be allowed (the command is in that notebook).

**Order here**: "Connect" cell → "Commands" → write missions.

**Main commands**:
- `arm_and_takeoff(5)` — ARM + take off to 5m (waits for the climb)
- `goto(north, east, alt)` — fly to a point (meters from start, alt is up)
- `land()` — land + disarm
- `where()` — where the drone is now
- `monitor(10)` — listen to telemetry for 10 sec

## 1. Connect (waits for a heartbeat from SITL)

In [ ]:
from pymavlink import mavutil
import time, threading, math

conn = mavutil.mavlink_connection('udpin:0.0.0.0:14550', source_system=255)
print("Listening on UDP 0.0.0.0:14550, waiting for heartbeat...")

def _hb_loop():
    while True:
        try:
            conn.mav.heartbeat_send(
                mavutil.mavlink.MAV_TYPE_GCS,
                mavutil.mavlink.MAV_AUTOPILOT_INVALID, 0, 0, 0)
        except Exception:
            pass
        time.sleep(1)

threading.Thread(target=_hb_loop, daemon=True).start()

msg = conn.recv_match(type='HEARTBEAT', blocking=True, timeout=60)
if not msg:
    raise SystemExit("No heartbeat in 60s — is SITL running? Is firewall 14550 allowed?")
print(f"ArduPilot online! sysid={conn.target_system}")

# Request the telemetry streams we need
for msg_id, interval_us in [(33, 200000), (32, 200000), (36, 200000)]:
    conn.mav.command_long_send(conn.target_system, 1,
        mavutil.mavlink.MAV_CMD_SET_MESSAGE_INTERVAL, 0,
        msg_id, interval_us, 0, 0, 0, 0, 0)
print("Telemetry streams requested (GLOBAL_POSITION_INT, LOCAL_POSITION_NED, SERVO)")

## 2. Commands (run once — then just call them)

In [ ]:
GUIDED, LAND_MODE = 4, 9

def _drain_status(t=0.0):
    """Prints accumulated STATUSTEXT (t sec)."""
    t0 = time.time()
    while time.time() - t0 <= t:
        m = conn.recv_match(type='STATUSTEXT', blocking=False)
        if m:
            print('  [AP]', m.text)
        else:
            time.sleep(0.05)

def set_mode(mode_num, name=""):
    conn.mav.command_long_send(conn.target_system, 1,
        mavutil.mavlink.MAV_CMD_DO_SET_MODE, 0,
        mavutil.mavlink.MAV_MODE_FLAG_CUSTOM_MODE_ENABLED, mode_num, 0, 0, 0, 0, 0)
    time.sleep(1)
    print(f"Mode -> {name or mode_num}")

def wait_ekf(timeout=60):
    """Wait for EKF readiness: once LOCAL_POSITION_NED starts, position is available.

    Without this, ARM fails with 'Need Position Estimate' / 'waiting for home'
    (EKF sets origin ~10-20 sec after SITL start).
    """
    print("Waiting for EKF readiness (origin + position)...")
    t0 = time.time()
    while time.time() - t0 < timeout:
        m = conn.recv_match(blocking=True, timeout=1)
        if not m:
            continue
        t = m.get_type()
        if t == 'STATUSTEXT':
            print('  [AP]', m.text)
        elif t == 'LOCAL_POSITION_NED':
            print("EKF ready: position is being published")
            time.sleep(2)  # small stabilization
            return True
    print("EKF wait timeout — trying ARM anyway")
    return False

def arm(attempts=3):
    """ARM with FORCE fallback, several cycles (arming checks 'Main loop slow' etc.)."""
    for cycle in range(attempts):
        for force in (False, True):
            label = 'FORCE ARM' if force else 'ARM'
            conn.mav.command_long_send(conn.target_system, 1,
                mavutil.mavlink.MAV_CMD_COMPONENT_ARM_DISARM, 0,
                1.0, 21196.0 if force else 0.0, 0, 0, 0, 0, 0)
            t0 = time.time()
            while time.time() - t0 < 5:
                m = conn.recv_match(blocking=True, timeout=1)
                if not m:
                    continue
                t = m.get_type()
                if t == 'STATUSTEXT':
                    print('  [AP]', m.text)
                elif t == 'HEARTBEAT' and (m.base_mode & mavutil.mavlink.MAV_MODE_FLAG_SAFETY_ARMED):
                    print(f"{label}: OK")
                    return True
                elif t == 'COMMAND_ACK' and m.command == mavutil.mavlink.MAV_CMD_COMPONENT_ARM_DISARM:
                    if m.result == 0:
                        print(f"{label}: OK (ACK)")
                        return True
                    break
        if cycle < attempts - 1:
            print(f"Attempt {cycle+1} failed, waiting 4s and retrying...")
            time.sleep(4)
    print("ARM failed")
    return False

def arm_and_takeoff(alt=5.0, timeout=60):
    """Wait EKF + ARM + NAV_TAKEOFF, waits for the climb."""
    set_mode(GUIDED, 'GUIDED')
    wait_ekf()
    if not arm():
        return False
    set_mode(GUIDED, 'GUIDED')  # mode may have reset after FORCE ARM
    time.sleep(2)               # auto_armed
    conn.mav.command_long_send(conn.target_system, 1,
        mavutil.mavlink.MAV_CMD_NAV_TAKEOFF, 0, 0, 0, 0, 0, 0, 0, alt)
    print(f"Taking off to {alt}m...")
    t0 = time.time()
    while time.time() - t0 < timeout:
        m = conn.recv_match(blocking=True, timeout=1)
        if not m:
            continue
        t = m.get_type()
        if t == 'STATUSTEXT':
            print('  [AP]', m.text)
        elif t == 'GLOBAL_POSITION_INT':
            a = m.relative_alt / 1000.0
            if a >= alt - 0.5:
                print(f"Altitude {a:.2f}m — takeoff complete")
                return True
    print("Takeoff timeout")
    return False

def where(quiet=False):
    """Drone position: (north, east, alt_up) meters from start."""
    m = conn.recv_match(type='LOCAL_POSITION_NED', blocking=True, timeout=3)
    if not m:
        print("No LOCAL_POSITION_NED (EKF has no origin yet?)")
        return None
    n, e, alt = m.x, m.y, -m.z
    if not quiet:
        print(f"Drone: north={n:.2f}m east={e:.2f}m alt={alt:.2f}m")
    return (n, e, alt)

def goto(north, east, alt, tolerance=0.7, timeout=60):
    """Fly to a point (meters from start; alt is height UP)."""
    conn.mav.set_position_target_local_ned_send(
        0, conn.target_system, 1,
        mavutil.mavlink.MAV_FRAME_LOCAL_NED,
        0b110111111000,          # position only
        north, east, -alt,
        0, 0, 0, 0, 0, 0, 0, 0)
    print(f"Flying to (north={north}, east={east}, alt={alt})...")
    t0 = time.time()
    while time.time() - t0 < timeout:
        p = where(quiet=True)
        if p is None:
            continue
        d = math.sqrt((p[0]-north)**2 + (p[1]-east)**2 + (p[2]-alt)**2)
        if d < tolerance:
            print(f"Arrived: north={p[0]:.2f} east={p[1]:.2f} alt={p[2]:.2f} (to target {d:.2f}m)")
            return True
        _drain_status(0)
        time.sleep(0.3)
    print("goto timeout — drone at", where(quiet=True))
    return False

def land(timeout=60):
    """Land (LAND mode), waits for disarm."""
    set_mode(LAND_MODE, 'LAND')
    t0 = time.time()
    while time.time() - t0 < timeout:
        m = conn.recv_match(type='HEARTBEAT', blocking=True, timeout=2)
        if m and not (m.base_mode & mavutil.mavlink.MAV_MODE_FLAG_SAFETY_ARMED):
            print("Landed and disarmed")
            return True
        _drain_status(0)
    print("Landing timeout")
    return False

def monitor(seconds=10):
    """Listen to telemetry for N seconds (STATUSTEXT + altitude)."""
    t0 = time.time()
    last_alt = None
    while time.time() - t0 < seconds:
        m = conn.recv_match(blocking=True, timeout=0.5)
        if not m:
            continue
        t = m.get_type()
        if t == 'STATUSTEXT':
            print('  [AP]', m.text)
        elif t == 'GLOBAL_POSITION_INT':
            a = m.relative_alt / 1000.0
            if last_alt is None or abs(a - last_alt) > 0.2:
                print(f"  [ALT] {a:.2f}m")
                last_alt = a

print("Commands ready: arm_and_takeoff, goto, land, where, monitor")

## Mission 1: take off to 1.7m and hover (inside a warehouse — ceiling and shelves!)

In [ ]:
arm_and_takeoff(1.7)
where()

## Mission 2: 2×2m square at 1.7m and land

(run after Mission 1 while the drone hovers; watch the viewport so the path doesn't run into a shelf — if it does, flip the signs/axes for your warehouse)

In [ ]:
goto(2, 0, 1.7)
goto(2, 2, 1.7)
goto(0, 2, 1.7)
goto(0, 0, 1.7)
land()

## Sandbox — write your own missions here

In [ ]:
# Example:
# arm_and_takeoff(3)
# goto(5, 0, 3)
# land()